# EP07 — LLM Sampling: Top-k, Top-p, Temperature
**Exam Relevance:** 2025 Q7.2 (10 marks — exact numbers given!) | 2024 Q6.3 (6 marks)

This question gives you a probability table and asks you to compute which tokens are selected.
**This is one of the most calculable questions in the exam — get it right.**

---

## The 2025 Q7.2 Exact Scenario

Input sequence: *"The dress colour is"*

| Token | red | blue | green | black | white | yellow | pink | purple | orange | gray |
|---|---|---|---|---|---|---|---|---|---|---|
| Prob | 0.35 | 0.30 | 0.15 | 0.08 | 0.05 | 0.03 | 0.02 | 0.01 | 0.005 | 0.005 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The exact 2025 exam data
tokens = ['red', 'blue', 'green', 'black', 'white', 'yellow', 'pink', 'purple', 'orange', 'gray']
probs  = [0.35,  0.30,   0.15,   0.08,   0.05,    0.03,    0.02,   0.01,    0.005,    0.005]

print("=" * 60)
print("2025 Q7.2 — Sampling Strategy Analysis")
print("=" * 60)

# --- Part a) Top-k sampling with k=2 ---
print("\n[a] Top-k sampling, k=2")
print("    Take the top 2 highest probability tokens:")
sorted_pairs = sorted(zip(probs, tokens), reverse=True)
top_k = [(t, p) for p, t in sorted_pairs[:2]]
for t, p in top_k:
    print(f"    {t}: {p}")
print(f"    Candidate set: {[t for t,p in top_k]}")

# --- Part b) Top-p sampling with p=0.85 ---
print("\n[b] Top-p (nucleus) sampling, p=0.85")
print("    Add tokens in descending order until cumulative prob >= 0.85:")
cumulative = 0
nucleus = []
for prob, token in sorted_pairs:
    cumulative += prob
    nucleus.append((token, prob, cumulative))
    print(f"    {token}: {prob:.3f} | cumulative: {cumulative:.3f}")
    if cumulative >= 0.85:
        break
print(f"    Candidate set: {[t for t,p,c in nucleus]}")
print(f"    (stopped at cumulative = {cumulative:.3f} >= 0.85)")

# --- Part c) Which considers entire distribution? ---
print("\n[c] Which method considers the entire distribution?")
print("    TOP-P considers the entire distribution.")
print("    Reason: The threshold p is compared against CUMULATIVE probability.")
print("    So how many tokens are included depends on the shape of the whole")
print("    distribution — if it's more spread out, more tokens are included.")
print("    Top-k always picks exactly k tokens regardless of distribution shape.")

In [ ]:
# Visualise the sampling strategies
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors_all = ['#4ECDC4'] * len(tokens)

# a) Top-k
colors_k = colors_all.copy()
colors_k[0] = '#FF6B6B'  # red
colors_k[1] = '#FF6B6B'  # blue

axes[0].bar(tokens, probs, color=colors_k, edgecolor='white')
axes[0].set_title('Top-k Sampling (k=2)\nCandidate: {red, blue}', fontsize=11)
axes[0].set_ylabel('Probability')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim(0, 0.45)
for i in range(2):
    axes[0].text(i, probs[i]+0.01, '✓', ha='center', fontsize=14, color='red')

# b) Top-p
cumulative_sum = np.cumsum(probs)
in_nucleus = cumulative_sum <= 0.85
# Include the first token that crosses 0.85
first_over = np.argmax(cumulative_sum >= 0.85)
in_nucleus[first_over] = True

colors_p = ['#FF6B6B' if inc else '#4ECDC4' for inc in in_nucleus]
axes[1].bar(tokens, probs, color=colors_p, edgecolor='white')
axes[1].set_title(f'Top-p Sampling (p=0.85)\nCandidate: {{red, blue, green, black}}', fontsize=11)
axes[1].set_ylabel('Probability')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0, 0.45)

# Add cumulative line
ax1b = axes[1].twinx()
ax1b.plot(tokens, cumulative_sum, 'ko-', markersize=4, linewidth=1.5, label='Cumulative prob')
ax1b.axhline(0.85, color='purple', linestyle='--', label='p=0.85 threshold')
ax1b.set_ylabel('Cumulative probability')
ax1b.set_ylim(0, 1.1)
ax1b.legend(loc='center right', fontsize=8)

# c) Compare k=2 vs p=0.85 on different distributions
# Show that top-p adapts but top-k doesn't
uniform_probs = [0.1] * 10  # flat distribution
peaked_probs = [0.9, 0.05] + [0.005]*8  # very peaked

axes[2].bar(tokens, uniform_probs, alpha=0.6, label='Uniform dist', color='orange')
axes[2].bar(tokens, peaked_probs, alpha=0.6, label='Peaked dist', color='green')
axes[2].set_title('Top-p Adapts to Distribution Shape\nTop-k always picks exactly k', fontsize=10)
axes[2].set_ylabel('Probability')
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

# Show how many tokens p=0.85 picks for each
for dist, color, y_pos in [(uniform_probs, 'orange', 0.15), (peaked_probs, 'green', 0.10)]:
    cum = np.cumsum(dist)
    n_picked = np.argmax(cum >= 0.85) + 1
    axes[2].text(7, y_pos, f'{color.title()}: p=0.85 picks {n_picked} tokens', 
                color=color, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## Temperature Sampling (2024 Q6.3)

**Temperature** controls the randomness of the output distribution:

$$P_T(w_i) = \frac{\exp(\text{logit}_i / T)}{\sum_j \exp(\text{logit}_j / T)}$$

- **T → 0:** Model becomes deterministic (always picks highest probability token)
- **T = 1:** Original distribution (no change)
- **T > 1:** Distribution flattens → more diverse, more random outputs
- **Higher T** → higher diversity but potentially less coherent text

**Top-P** (nucleus sampling):
- Dynamically selects how many tokens to include based on cumulative probability
- **Higher p** → more tokens in nucleus → more diverse outputs

In [ ]:
import torch
import torch.nn.functional as F

# Show effect of temperature on the exam probability distribution
# First, back-compute approximate logits from the probabilities
probs_t = torch.tensor(probs, dtype=torch.float)
logits = torch.log(probs_t)  # approximate logits

temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]
fig, axes = plt.subplots(1, len(temperatures), figsize=(18, 4))

for ax, T in zip(axes, temperatures):
    scaled = logits / T
    probs_new = F.softmax(scaled, dim=0).numpy()
    bars = ax.bar(tokens, probs_new, color='steelblue', edgecolor='white')
    ax.set_title(f'T = {T}', fontsize=11)
    ax.set_ylabel('Probability' if T == temperatures[0] else '')
    ax.tick_params(axis='x', rotation=60)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 1.05)
    entropy = -np.sum(probs_new * np.log(probs_new + 1e-10))
    ax.set_xlabel(f'Entropy: {entropy:.2f}', fontsize=8)

plt.suptitle('Effect of Temperature on Token Probability Distribution\nLow T = deterministic | High T = random', 
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("Key observations:")
print("T=0.1: Almost all probability on 'red' — very deterministic")
print("T=1.0: Original distribution unchanged")
print("T=5.0: Very flat — nearly uniform — very random")

## Exam Quick-Reference Summary

**Top-k (k=2):** Simply take the k=2 highest probability tokens → {red, blue}

**Top-p (p=0.85):** Sort descending, add until cumulative ≥ 0.85:
- red: 0.35 → cum=0.35
- blue: 0.30 → cum=0.65  
- green: 0.15 → cum=0.80
- black: 0.08 → cum=0.88 ✓ stop
- **Candidate set: {red, blue, green, black}**

**Part c) Which considers entire distribution? → Top-p**
- Top-k: always exactly k tokens, ignores distribution shape
- Top-p: number of tokens varies based on how spread the distribution is — implicitly considers the whole distribution

**Temperature (2024):**
- Higher T → flatter distribution → more diverse outputs
- Higher Top-p → larger nucleus → more diverse outputs